<a href="https://colab.research.google.com/github/swarnapatro/AI_KickStart/blob/main/customer_support_lcel_ollama.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎧 Customer Support Assistant with LangChain (LCEL) + Local Ollama

**Hands-on training notebook – no coding background needed.**

### What you will build
A small "AI support desk" that can
1. Write a polite reply to a customer message
2. Sort (classify) tickets into Billing / Technical / Shipping / Other
3. Judge how urgent and how unhappy the customer is
4. Combine all of this into **one automatic pipeline**

### The one big idea: the pipe `|`
LangChain Expression Language (**LCEL**) lets us connect small building blocks with a pipe, just like an assembly line:

```
 customer message  →  PROMPT  |  LLM  |  PARSER  →  final answer
                      (recipe)   (brain)  (clean-up)
```

* **Prompt** – a template (recipe) where we drop in the customer's text
* **LLM** – the AI model running **locally on your laptop** through Ollama (no internet, no API key, no cost)
* **Parser** – turns the AI's raw output into plain text we can use

The output of the block on the left of `|` becomes the input of the block on the right.

### How to use this notebook
Run the cells **from top to bottom** (click a cell → press `Shift + Enter`). Every code cell starts with a comment box explaining **what it does, its input and its output**. Lines starting with `#` are comments – the computer ignores them; they are notes for humans.

### Before you start
1. Ollama must be installed and running (check: open a terminal and type `ollama list`).
2. A model must be downloaded, for example: `ollama pull llama3.2`

## Step 0 – Install the libraries (one-time)

In [ ]:
# ============================================================
# CELL PURPOSE : Install the Python libraries this notebook needs.
# INPUT        : nothing
# OUTPUT       : packages installed (you may see "Requirement already
#                satisfied" – that is fine, it means you already have them)
# NOTE         : Only needs to be run ONCE per laptop.
# ============================================================

# "%pip" = install a package from inside the notebook. "-q" = quiet (less text).
#   langchain-core   -> the LCEL building blocks (prompts, parsers, the | pipe)
#   langchain-ollama -> the connector that lets LangChain talk to Ollama
%pip install -q langchain-core langchain-ollama

## Step 1 – Import the tools we need
Think of imports as *taking tools out of a toolbox* before starting work.

In [ ]:
# ============================================================
# CELL PURPOSE : Load (import) the LangChain tools used in this notebook.
# INPUT        : nothing
# OUTPUT       : nothing printed – if there is no error, all tools loaded OK
# ============================================================

# ChatOllama = the "connector" to the AI model running locally in Ollama.
# WHY: without it, LangChain does not know how to talk to our local model.
from langchain_ollama import ChatOllama

# ChatPromptTemplate = a fill-in-the-blanks message template.
# WHY: we write the instructions once and reuse them for every customer.
#      Example: "Customer says: {message}" -> {message} is replaced each time.
from langchain_core.prompts import ChatPromptTemplate

# StrOutputParser = converts the model's answer object into a plain text string.
# WHY: the model returns a "message object" (text + extra details).
#      We only want the text.
from langchain_core.output_parsers import StrOutputParser

# JsonOutputParser = converts text that looks like JSON into a Python dictionary.
# WHY: lets us get several answers at once (e.g. urgency AND sentiment)
#      in a tidy "label: value" format.
from langchain_core.output_parsers import JsonOutputParser

# RunnableLambda = wraps our own small Python function so it can sit inside a "|" chain.
# WHY: lets us add custom steps (like cleaning text) to the assembly line.
# RunnablePassthrough = a block that simply passes its input through unchanged
#      (and, with .assign(), can ADD new fields next to it).
# WHY: lets us keep the original customer message while also adding new results.
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

print("✅ All tools imported successfully")

✅ All tools imported successfully


In [ ]:
# Cell A: install Ollama on the Colab machine
!apt-get install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
# Cell B: start the Ollama server in the background and download the model
import subprocess, time
subprocess.Popen(["ollama", "serve"])
time.sleep(5)                      # give the server a few seconds to start
!ollama pull llama3.2

## Step 2 – Connect to the local AI model (LLM)

In [ ]:
# ============================================================
# CELL PURPOSE : Create the "brain" (LLM) of our support desk.
# INPUT        : the model name installed in your Ollama
# OUTPUT       : a variable called `llm` that we can use in every chain.
#                We also send a quick "hello" to prove the connection works.
# ============================================================

# The name of the model downloaded in Ollama. (Run `ollama list` to see yours.)
# You can change this to "llama3.1:8b" (smarter but slower) or "qwen3:4b".
MODEL_NAME = "llama3.2"

llm = ChatOllama(
    model=MODEL_NAME,            # which local model to use
    temperature=0,               # 0 = steady, predictable answers (best for support).
                                 # Higher (e.g. 0.8) = more creative / more random.
    base_url="http://localhost:11434",  # where Ollama runs on this laptop (default address)
)

# Quick connection test: send a plain question straight to the model.
# .invoke(...) means "run it now with this input".
test = llm.invoke("Say hello to a customer in one short sentence.")

# test is a message object; .content holds just the text.
print("Model replied:", test.content)
# Example output (wording will vary): Model replied: Hello! Welcome, how can I help you today?

Model replied: Hello, welcome to our store!


## Step 3 – Your FIRST chain: Prompt `|` LLM `|` Parser
We build the assembly line:  **recipe → brain → clean-up**

In [ ]:
# ============================================================
# CELL PURPOSE : Build Chain #1 – "Write a reply to a customer".
# INPUT (of the chain) : a dictionary like {"message": "<customer text>"}
# OUTPUT (of the chain): a plain text string = the support reply
# ============================================================

# --- Block 1: the PROMPT (recipe) -------------------------------------------
# "system" message = permanent rules for the AI / LLM (its job and behaviour).
# "human"  message = the customer's text. {message} is a blank to be filled in.
reply_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a friendly and professional customer support agent for an online "
     "shop called 'ShopEasy'. Reply in 3-4 short sentences. Apologise if the "
     "customer has a problem, explain the next step, and never invent order "
     "numbers or policies."),
    ("human", "Customer message: {message}"),
])

# --- Block 2: the LLM (brain) -> already created in Step 2 as `llm` ----------

# --- Block 3: the PARSER (clean-up) -----------------------------------------
parser = StrOutputParser()

# --- THE CHAIN: connect the blocks with the pipe symbol "|" ------------------
# Data flows left -> right:  prompt fills the blanks | llm thinks | parser cleans
reply_chain = reply_prompt | llm | parser

# --- Run it -----------------------------------------------------------------
customer_message = "Hi, my order arrived yesterday but the item are not working. I want my money back."

# .invoke() runs the chain. The dictionary key "message" must match {message} above.
answer = reply_chain.invoke({"message": customer_message})

print("CUSTOMER:", customer_message)
print()
print("AI REPLY:", answer)
# Example output: a polite 3-4 sentence apology that explains the refund/return steps.

CUSTOMER: Hi, my order arrived yesterday but the item are not working. I want my money back.

AI REPLY: I apologize for the inconvenience you're experiencing with your order. I'd be happy to help you with a refund. Could you please provide me with your order details, such as the items that are not working and the date you placed the order? I'll look into this further and guide you on the next steps to process a refund.


## Step 4 – Look inside the pipe (what happens at each stage?)
The pipe is not magic. Let us run each block **one by one** and watch the data change shape.

In [ ]:
# ============================================================
# CELL PURPOSE : Run each block of Chain #1 separately to SEE what
#                goes in and what comes out of every stage.
# INPUT        : the same customer message as before
# OUTPUT       : 3 printed stages (prompt messages -> AI message object -> plain text)
# ============================================================

test_input = {"message": "Where is my parcel? It was supposed to arrive 3 days ago."}

# STAGE 1 – PROMPT
# Input : {"message": "..."}            (a dictionary)
# Output: the full list of chat messages with the blank filled in
stage1 = reply_prompt.invoke(test_input)
print("STAGE 1 – after PROMPT (blank filled in):")
for m in stage1.messages:
    print(f"   [{m.type}] {m.content}")

# STAGE 2 – LLM
# Input : the messages from stage 1
# Output: an AI message object (contains the text plus technical details)
stage2 = llm.invoke(stage1)
print("\nSTAGE 2 – after LLM (raw model object type):", type(stage2).__name__)
print("   text inside:", stage2.content)

# STAGE 3 – PARSER
# Input : the AI message object from stage 2
# Output: just the plain text string
stage3 = parser.invoke(stage2)
print("\nSTAGE 3 – after PARSER (now just plain text, no extra details):")
print("  ", str(stage3))
# Output: the same sentence as stage 2, but as simple text that is easy to use.

STAGE 1 – after PROMPT (blank filled in):
   [system] You are a friendly and professional customer support agent for an online shop called 'ShopEasy'. Reply in 3-4 short sentences. Apologise if the customer has a problem, explain the next step, and never invent order numbers or policies.
   [human] Customer message: Where is my parcel? It was supposed to arrive 3 days ago.

STAGE 2 – after LLM (raw model object type): AIMessage
   text inside: I'm so sorry to hear that you're experiencing a delay with your parcel. I'd be happy to help you track the status of your order. Can you please provide me with your order number or the email address you used to place the order, so I can look into this further for you? I'll do my best to provide you with an update on the expected delivery date.

STAGE 3 – after PARSER (now just plain text, no extra details):
   I'm so sorry to hear that you're experiencing a delay with your parcel. I'd be happy to help you track the status of your order. Can you p

## Step 5 – Chain #2: Classify the ticket (Billing / Technical / Shipping / Other)
Same pattern, different recipe. Support teams use this to send tickets to the right department.

In [ ]:
# ============================================================
# CELL PURPOSE : Build Chain #2 – decide which department should handle a ticket.
# INPUT        : {"message": "<customer text>"}
# OUTPUT       : ONE word: Billing, Technical, Shipping or Other
# ============================================================

classify_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a ticket router for a customer support team. "
     "Classify the customer message into EXACTLY ONE of these categories: "
     "Billing, Technical, Shipping, Other.\n"
     "Billing = payments, refunds, invoices, charges.\n"
     "Technical = product not working, app or website problems.\n"
     "Shipping = delivery, tracking, late or lost parcels.\n"
     "Other = anything else.\n"
     "Answer with the category word ONLY. No explanation."),
    ("human", "{message}"),
])

# A tiny custom step (our own Python function) to tidy the model's answer.
# Input : text such as " billing.\n"   Output: "Billing"
def clean_category(text: str) -> str:
    allowed = ["Billing", "Technical", "Shipping", "Other"]
    for word in allowed:                       # look for each allowed word...
        if word.lower() in text.lower():       # ...ignoring capital/small letters
            return word                        # ...and return the first match
    return "Other"                             # safety net if the model says something odd

# Chain with FOUR blocks: prompt | llm | parser | our own cleaning function.
# RunnableLambda(...) is what lets a normal function join the pipe.
classify_chain = classify_prompt | llm | StrOutputParser() | RunnableLambda(clean_category)

# --- Try it on 4 different messages -----------------------------------------
samples = [
    "I was charged twice for my order this month!",
    "The mobile app crashes every time I open it.",
    "My parcel has been stuck at the courier for a week.",
    "Do you have a store in Chennai?",
]
for text in samples:
    category = classify_chain.invoke({"message": text})
    print(f"{category:<10} <- {text}")
# Example output (one line per message):
# Billing    <- I was charged twice for my order this month!
# Technical  <- The mobile app crashes every time I open it.
# Shipping   <- My parcel has been stuck at the courier for a week.
# Other      <- Do you have a store in Chennai?

Billing    <- I was charged twice for my order this month!
Technical  <- The mobile app crashes every time I open it.
Shipping   <- My parcel has been stuck at the courier for a week.
Other      <- Do you have a store in Chennai?


## Step 6 – Chain #3: Urgency and customer mood (structured answer)
Instead of one word, we ask for **several fields** and get them back as a Python dictionary.

In [ ]:
# ============================================================
# CELL PURPOSE : Build Chain #3 – measure URGENCY and customer SENTIMENT.
# INPUT        : {"message": "<customer text>"}
# OUTPUT       : a dictionary, e.g. {"urgency": "High", "sentiment": "Angry"}
# ============================================================

analysis_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You analyse customer support messages. "
     "Return ONLY a JSON object with exactly two keys:\n"
     '"urgency"   -> one of: Low, Medium, High\n'
     '"sentiment" -> one of: Happy, Neutral, Angry\n'
     "No extra text, no explanation."),
    ("human", "{message}"),
])

# JsonOutputParser converts the model's JSON-looking text into a real dictionary.
# Input to parser: the text '{"urgency": "High", "sentiment": "Angry"}'
# Output of parser: {'urgency': 'High', 'sentiment': 'Angry'}
analysis_chain = analysis_prompt | llm | JsonOutputParser()

# --- Try it -----------------------------------------------------------------
result = analysis_chain.invoke(
    {"message": "This is the THIRD time my payment failed! I need this fixed TODAY or I cancel!"}
)

print("Full result  :", result)
print("Urgency only :", result["urgency"])     # pick one value out of the dictionary
print("Sentiment    :", result["sentiment"])
# Example output:
# Full result  : {'urgency': 'High', 'sentiment': 'Angry'}
# Urgency only : High
# Sentiment    : Angry

Full result  : {'urgency': 'High', 'sentiment': 'Angry'}
Urgency only : High
Sentiment    : Angry


## Step 7 – Chain #4: The complete pipeline (all chains working together)
Now we connect **classification + analysis + reply** into one automatic desk.
The reply will change depending on the department and how upset the customer is.

In [ ]:
# ============================================================
# CELL PURPOSE : Build the FULL support pipeline from the smaller chains.
# INPUT        : {"message": "<customer text>"}
# OUTPUT       : a dictionary with 4 fields:
#                message, category, analysis, reply
# ============================================================

# A smarter reply recipe: it now also receives the category and mood,
# so the AI can adjust its tone and mention the right department.
smart_reply_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a friendly, professional support agent for 'ShopEasy'. "
     "This ticket is routed to the {category} team. "
     "Customer urgency: {urgency}. Customer mood: {sentiment}. "
     "Write a reply in 3-4 short sentences. If the mood is Angry, start with a "
     "sincere apology. If urgency is High, promise that the {category} team will "
     "prioritise the case. Never invent order numbers or policies."),
    ("human", "{message}"),
])

# Helper function: takes the big dictionary and builds the input the reply prompt needs.
# Input : {"message": ..., "category": ..., "analysis": {"urgency":..., "sentiment":...}}
# Output: {"message": ..., "category": ..., "urgency": ..., "sentiment": ...}
def prepare_reply_input(data: dict) -> dict:
    return {
        "message":   data["message"],
        "category":  data["category"],
        "urgency":   data["analysis"]["urgency"],
        "sentiment": data["analysis"]["sentiment"],
    }

# THE FULL PIPELINE – read it top to bottom like a recipe:
support_pipeline = (
    # 1) Start with {"message": ...} and ADD a new field "category" by running classify_chain
    RunnablePassthrough.assign(category=classify_chain)
    # 2) ADD another field "analysis" by running analysis_chain
    | RunnablePassthrough.assign(analysis=analysis_chain)   # analysis = {"urgency": "High", "sentiment": "Angry"}
    # 3) ADD the final field "reply": reshape the data, then prompt | llm | parser
    | RunnablePassthrough.assign(
        reply=RunnableLambda(prepare_reply_input) | smart_reply_prompt | llm | StrOutputParser()
    )
)

# --- Try it -----------------------------------------------------------------
ticket = {"message": "I was charged twice for my order and nobody is answering my emails. Fix this today!"}
output = support_pipeline.invoke(ticket)

print("MESSAGE :", output["message"])
print("CATEGORY:", output["category"])
print("ANALYSIS:", output["analysis"])
print("\nREPLY   :", output["reply"])
# Example output:
# CATEGORY: Billing
# ANALYSIS: {'urgency': 'High', 'sentiment': 'Angry'}
# REPLY   : a short apology saying the Billing team will prioritise the double charge.

MESSAGE : I was charged twice for my order and nobody is answering my emails. Fix this today!
CATEGORY: Billing
ANALYSIS: {'urgency': 'High', 'sentiment': 'Angry'}

REPLY   : I'm so sorry to hear that you're experiencing this issue and that our team hasn't been able to assist you via email. I'm here to help and I'm going to personally look into this for you. We'll prioritize your case and work to resolve the double charge as soon as possible. Can you please confirm your order number so I can expedite the process?


## Step 8 – Process many tickets at once with `.batch()`
A real support desk receives many tickets. `.batch()` runs the same chain on a whole list.

In [ ]:
# ============================================================
# CELL PURPOSE : Run the full pipeline on several tickets in one go
#                and show a neat summary.
# INPUT        : a LIST of dictionaries, one per ticket
# OUTPUT       : a LIST of result dictionaries (same order as the input),
#                printed as a small table
# ============================================================

tickets = [
    {"message": "I love the new jacket, thank you! Just wondering if it comes in blue."},
    {"message": "The website keeps logging me out when I try to pay. Very frustrating."},
    {"message": "My order #12345 never arrived and it has been two weeks. I want a refund NOW."},
]

# .batch() = run the chain once per item in the list (several at the same time).
results = support_pipeline.batch(tickets)

# Print a simple summary table.
print(f"{'CATEGORY':<10} {'URGENCY':<8} {'MOOD':<8} MESSAGE")
print("-" * 80)
for r in results:
    print(f"{r['category']:<10} {r['analysis']['urgency']:<8} {r['analysis']['sentiment']:<8} {r['message'][:45]}...")

# Show the AI reply for the last ticket as an example
print("\nSample reply for the last ticket:\n", results[-1]["reply"])
# Example output: 3 table rows (one per ticket) followed by a sample reply.

CATEGORY   URGENCY  MOOD     MESSAGE
--------------------------------------------------------------------------------
Other      Low      Happy    I love the new jacket, thank you! Just wonder...
Technical  High     Angry    The website keeps logging me out when I try t...
Shipping   High     Angry    My order #12345 never arrived and it has been...

Sample reply for the last ticket:
 I'm so sorry to hear that your order has not arrived and you're experiencing a delay. I'm here to help and want to assure you that we're taking immediate action to resolve this issue. Our Shipping team will prioritize your case to ensure a swift resolution, and I'll personally follow up with them to expedite the process. Please allow me to assist you further with a refund or alternative solution.


## Step 9 – Watch the answer appear word by word with `.stream()`
Just like ChatGPT: instead of waiting for the full answer, show it as it is being written.

In [ ]:
# ============================================================
# CELL PURPOSE : Stream a reply piece by piece (better user experience).
# INPUT        : {"message": "<customer text>"}
# OUTPUT       : the reply printed gradually on the screen
# ============================================================

# .stream() gives us small pieces ("chunks") as soon as the model produces them,
# instead of waiting for the whole answer like .invoke() does.
for chunk in reply_chain.stream({"message": "How do I reset my password? I cannot log in."}):
    print(chunk, end="", flush=True)   # end="" keeps all pieces on the same line
print()  # move to a new line at the end
# Example output: a short helpful answer appearing in small pieces.

I'm so sorry to hear that you're having trouble logging in. To reset your password, please click on the "Forgot Password" link on our website's login page, and follow the prompts to enter your email address associated with your account. We'll send you a password reset link to the email address you provided. If you have any further issues, please don't hesitate to contact us.


## Step 10 – Try it yourself!
Type your own customer message and see the whole pipeline respond.

In [ ]:
# ============================================================
# CELL PURPOSE : Let YOU test the pipeline with your own message.
# INPUT        : change the text inside my_message below, then run the cell
# OUTPUT       : category, urgency, mood and the AI reply
# ============================================================

# ✏️ EDIT THIS LINE – write any customer complaint or question:
my_message = "The charger I bought stopped working after two days. Can I get a replacement?"

out = support_pipeline.invoke({"message": my_message})

print("Category :", out["category"])
print("Urgency  :", out["analysis"]["urgency"])
print("Mood     :", out["analysis"]["sentiment"])
print("\nReply:\n", out["reply"])

Category : Technical
Urgency  : High
Mood     : Angry

Reply:
 I'm so sorry to hear that your charger stopped working after just two days. I'd be happy to help you with a replacement. Our Technical team will look into this for you and get back to you as soon as possible. We'll do our best to resolve this issue for you as quickly as possible.


## 🎓 Recap

| Concept | In plain words | Code you used |
|---|---|---|
| Prompt | Recipe with blanks | `ChatPromptTemplate` |
| LLM | The AI brain (local) | `ChatOllama` |
| Parser | Cleans the output | `StrOutputParser`, `JsonOutputParser` |
| Pipe `\|` | Connect blocks into an assembly line | `prompt \| llm \| parser` |
| `.invoke()` | Run once | |
| `.batch()` | Run for a list | |
| `.stream()` | Show answer gradually | |

## ✏️ Practice exercises (no heavy coding needed)
1. In **Step 2**, change `MODEL_NAME` to `"llama3.1:8b"` and compare reply quality.
2. In **Step 3**, change the shop name or the tone (e.g. *very formal*, *very casual*).
3. In **Step 5**, add a new category such as `Returns` (add it to the `allowed` list and the prompt).
4. In **Step 2**, set `temperature=0.9` and run Step 3 twice – what changes?
5. Write your own chain: *prompt \| llm \| parser* that **summarises** a long complaint in one sentence.